In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from config import config
from data import load_data, apply_fe, get_folds, ALL_FEAT, TARGET
from train import run_cv, fit_predict
from utils import set_seed, save_results

import warnings
warnings.filterwarnings('ignore')


In [4]:
set_seed(config.general.SEED)
train_df, test_df = load_data(config)
train_fe = apply_fe(train_df)
test_fe = apply_fe(test_df)

X, y = train_fe[ALL_FEAT], train_fe[TARGET]
X_test = test_fe[ALL_FEAT]

for n_splits in [1, 5]:
    config.cv.n_splits = n_splits
    folds = get_folds(X, y, config)

    results = []
    for model_name in ["baseline", "knn", "tree", "rf", "catboost", "lightgbm", "xgboost"]:
        run_cv(model_name, X, y, folds, config, results)

    df = save_results(results, Path(config.paths.RESULTS_DIR) / f"boosting2_cv_results_n{n_splits}.csv")
    print(f"--- n_splits={n_splits} ---")
    print(df.groupby("model").mean(numeric_only=True))


--- n_splits=1 ---
          fold  accuracy   roc_auc
model                             
baseline   0.0  0.804469  0.851976
catboost   0.0  0.782123  0.857378
knn        0.0  0.765363  0.843215
lightgbm   0.0  0.804469  0.857708
rf         0.0  0.782123  0.865679
tree       0.0  0.776536  0.845652
xgboost    0.0  0.787709  0.867852
--- n_splits=5 ---
          fold  accuracy   roc_auc
model                             
baseline   2.0  0.828272  0.870568
catboost   2.0  0.831643  0.876179
knn        2.0  0.817042  0.863715
lightgbm   2.0  0.820426  0.878330
rf         2.0  0.833890  0.875991
tree       2.0  0.814801  0.864606
xgboost    2.0  0.830525  0.880422


In [8]:
print(df.groupby("model").agg(["mean", "std"]))

print(df.groupby("model")["accuracy"].mean().sort_values(ascending=False))

         fold            accuracy             roc_auc          
         mean       std      mean       std      mean       std
model                                                          
baseline  2.0  1.581139  0.828272  0.024767  0.870568  0.015566
catboost  2.0  1.581139  0.831643  0.031323  0.876179  0.012610
knn       2.0  1.581139  0.817042  0.016402  0.863715  0.011854
lightgbm  2.0  1.581139  0.820426  0.013769  0.878330  0.014598
rf        2.0  1.581139  0.833890  0.028840  0.875991  0.012223
tree      2.0  1.581139  0.814801  0.034713  0.864606  0.022072
xgboost   2.0  1.581139  0.830525  0.031139  0.880422  0.012093
model
rf          0.833890
catboost    0.831643
xgboost     0.830525
baseline    0.828272
lightgbm    0.820426
knn         0.817042
tree        0.814801
Name: accuracy, dtype: float64


In [5]:
BEST_MODEL = "rf"

preds = fit_predict(BEST_MODEL, X, y, X_test, config)
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": preds.astype(int),
})
submission.to_csv(Path(config.paths.RESULTS_DIR)/ f"submission_{BEST_MODEL}.csv", index=False)

submission.head()
print(submission.shape)
print(submission.columns)

(418, 2)
Index(['PassengerId', 'Survived'], dtype='str')


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline

from data import preprocessor
from train import MODEL_REGISTRY

PARAM_GRIDS = {
    "knn": {"model__n_neighbors": [3, 5, 7, 11]},
    "tree": {"model__max_depth": [3, 5, 10, None]},
    "rf": {
        "model__n_estimators": [100, 200, 300],
        "model__max_depth": [None, 5, 10],
    },
    "catboost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__depth": [4, 6, 8],
    },
    "lightgbm": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [-1, 5, 10],
        "model__num_leaves": [15, 31, 63],
    },
    "xgboost": {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__max_depth": [3, 6, 9],
    },
}

SCORING = "accuracy" if config.general.TASK == "classification" else "neg_root_mean_squared_error"

best_params = {}

for model_name, param_grid in PARAM_GRIDS.items():
    model_cls, model_family = MODEL_REGISTRY[model_name][config.general.TASK]

    try:
        base_model = model_cls(random_state=config.general.SEED)
    except TypeError:
        base_model = model_cls()

    pipeline = Pipeline([
        ("preprocess", preprocessor(model_family)),
        ("model", base_model),
    ])

    # search = GridSearchCV(pipeline, param_grid=param_grid, cv=folds, scoring=SCORING, n_jobs=-1)
    search = RandomizedSearchCV(
        pipeline, param_distributions=param_grid, n_iter=20, cv=folds,
        scoring=SCORING, random_state=config.general.SEED, n_jobs=-1
    )
    search.fit(X, y)

    best_params[model_name] = search.best_params_
    print(f"{model_name}: best_score={search.best_score_:.4f}, best_params={search.best_params_}")

knn: best_score=0.8170, best_params={'model__n_neighbors': 11}
tree: best_score=0.8148, best_params={'model__max_depth': 3}
rf: best_score=0.8339, best_params={'model__n_estimators': 200, 'model__max_depth': 5}
0:	learn: 0.6321383	total: 53.8ms	remaining: 16.1s
1:	learn: 0.5791012	total: 54.1ms	remaining: 8.05s
2:	learn: 0.5440662	total: 54.3ms	remaining: 5.38s
3:	learn: 0.5130394	total: 54.6ms	remaining: 4.04s
4:	learn: 0.4867296	total: 54.8ms	remaining: 3.23s
5:	learn: 0.4659003	total: 55.1ms	remaining: 2.7s
6:	learn: 0.4497987	total: 55.6ms	remaining: 2.33s
7:	learn: 0.4347692	total: 56ms	remaining: 2.04s
8:	learn: 0.4260873	total: 56.2ms	remaining: 1.82s
9:	learn: 0.4186137	total: 56.5ms	remaining: 1.64s
10:	learn: 0.4095531	total: 56.7ms	remaining: 1.49s
11:	learn: 0.4015417	total: 56.9ms	remaining: 1.37s
0:	learn: 0.6399387	total: 54ms	remaining: 16.2s
12:	learn: 0.3920416	total: 57.2ms	remaining: 1.26s
13:	learn: 0.3867022	total: 57.4ms	remaining: 1.17s
1:	learn: 0.5873991	total